# 07 ? Piloto CPU: curriculum e workers

Piloto t?cnico explorat?rio para conferir gera??o online, adapta??o para o contrato Chronos-2 e igualdade entre batches gerados diretamente e por DataLoader multiprocessado.

A verifica??o de configura??o testa tr?s est?gios crescentes (canais e n?mero m?ximo de operadores). Para testar execu??o e workers, usa-se uma configura??o est?vel sem operadores: o gerador atual n?o garante estabilidade conjunta para sistemas mistos aleat?rios.

Registro: seed expl?cita 8217; CPU; tamanhos reduzidos; este piloto n?o mede throughput de treino nem stall de GPU.

## Reproduction instructions

Use Python 3.11 from the repository root. Install the project dependencies with `python -m pip install -e ".[learning,dev,release]"`, then execute this notebook with:

`python -m nbconvert --to notebook --execute --inplace experiments/generator_model_evaluation/07_cpu_pilot_curriculum_workers.ipynb`

The Chronos-2 checkpoint used by this experiment is pinned to `amazon/chronos-2@29ec3766d36d6f73f0696f85560a422f50e8498c`; the skeleton teacher, where used, is pinned to `mateuswiteck/tsfmxai-canonicalization-encoder-decoder` variant `transformer_skeleton` at `2fcdcc88b41125bcfd92953c0bb8a668a603d167`. The notebook records its own seeds, settings, observed result, and current limitations.


In [1]:
from pathlib import Path
import sys, time
import numpy as np
import torch
ROOT=Path.cwd()
while not (ROOT/'src/tsfmxai').exists() and ROOT!=ROOT.parent: ROOT=ROOT.parent
if not (ROOT/'src/tsfmxai').exists(): raise RuntimeError('repository root not found')
sys.path.insert(0,str(ROOT/'src'))
from tsfmxai.datasets.online import OnlineBatchPlan,CurriculumStage,make_dataloader
from tsfmxai.datasets.multiseries import TrainingFormat,TrajectoryConfig
from tsfmxai.generators.initialization import HistorySpec
from tsfmxai.generators.combinatorial_expression import CombinatorialTreeConfig
from tsfmxai.generators.multiseries import MultiseriesConfig,NoiseConfig
from tsfmxai.datasets.chronos2_adapter import adapt_tsfmxai_batch
SEED=8217; EXAMPLES=8; CONTEXT=32; HORIZON=16; PATCH=8
def make_config(channels, ops):
 return MultiseriesConfig(ast_config=CombinatorialTreeConfig(minimum_operators=ops,maximum_operators=ops,minimum_lag=1,maximum_lag=2,max_degree=2,require_lag=ops>0),
  channel_count_probs={channels:1.0},dependency_probs={'independent':1.0},treatment_probs={'original':1.0},
  max_external_sources=1,allow_no_lags=ops==0,noise=NoiseConfig(std_range=(0.02,0.05)))
stages=(CurriculumStage(0,make_config(2,0),EXAMPLES),CurriculumStage(2,make_config(4,1),EXAMPLES),CurriculumStage(4,make_config(6,2),EXAMPLES))
format=TrainingFormat(CONTEXT,HORIZON,HORIZON,batch_size=EXAMPLES,dtype='float32')
trajectory=TrajectoryConfig(CONTEXT+HORIZON*3,HistorySpec('uniform','uniform'))
plan=OnlineBatchPlan(stages,6,SEED,trajectory,format)
print('torch',torch.__version__,'CUDA',torch.cuda.is_available(),'stages',[(s.first_batch,next(iter(s.config.channel_count_probs)),s.config.ast_config.maximum_operators) for s in stages])

torch 2.12.0+cpu CUDA False stages [(0, 2, 0), (2, 4, 1), (4, 6, 2)]


## Progress?o de curriculum declarada

Confere que o plano seleciona os canais e operadores configurados em cada fronteira. A gera??o de operador alto n?o ? executada neste smoke test, pois os mecanismos aleat?rios n?o t?m certificado conjunto de estabilidade.

In [2]:
expected=[(0,2,0),(2,4,1),(4,6,2)]
observed=[]
for batch_id in (0,2,4):
 stage=plan.stage_for(batch_id)
 row=(batch_id,next(iter(stage.config.channel_count_probs)),stage.config.ast_config.maximum_operators)
 observed.append(row)
assert observed==expected
print('curriculum boundaries (batch, channels, max_ops):',observed)
# Generate one real batch with a stable constant-plus-innovation system.
safe=OnlineBatchPlan((CurriculumStage(0,make_config(6,0),EXAMPLES),),1,SEED+1,
 TrajectoryConfig(CONTEXT+HORIZON*3,HistorySpec('uniform','uniform')),format)
raw=safe.generate(0)
adapted=adapt_tsfmxai_batch(raw,layout='batch_time_channel',output_patch_size=PATCH)
assert adapted['context'].shape==(EXAMPLES*6,CONTEXT)
assert adapted['future_target'].shape==(EXAMPLES*6,HORIZON)
assert adapted['noise_std'].shape==(EXAMPLES*6,)
assert adapted['num_output_patches']==2
print('safe batch',raw.context.shape,'Chronos rows',tuple(adapted['context'].shape),'patches',adapted['num_output_patches'])

curriculum boundaries (batch, channels, max_ops): [(0, 2, 0), (2, 4, 1), (4, 6, 2)]
safe batch (8, 32, 6) Chronos rows (48, 32) patches 2


## Reprodutibilidade com DataLoader

Compara batches por ID global, primeiro em acesso direto e depois em dois workers CPU. O resultado deve ser id?ntico, mesmo que os workers processem IDs em processos diferentes.

In [3]:
# Windows DataLoader spawn requer um arquivo main import?vel.
from pathlib import Path
script = ROOT/'outputs'/'generator_model_evaluation'/'_pilot_07_worker_check.py'
script.parent.mkdir(parents=True, exist_ok=True)
worker_source = "from pathlib import Path\nimport sys, numpy as np\nroot=Path(__file__).resolve().parents[2]\nsys.path.insert(0,str(root/'src'))\nfrom tsfmxai.datasets.online import OnlineBatchPlan,CurriculumStage,make_dataloader\nfrom tsfmxai.datasets.multiseries import TrainingFormat,TrajectoryConfig\nfrom tsfmxai.generators.initialization import HistorySpec\nfrom tsfmxai.generators.combinatorial_expression import CombinatorialTreeConfig\nfrom tsfmxai.generators.multiseries import MultiseriesConfig,NoiseConfig\ndef make_config(channels):\n return MultiseriesConfig(ast_config=CombinatorialTreeConfig(minimum_operators=0,maximum_operators=0,minimum_lag=1,maximum_lag=2,max_degree=2,require_lag=False),channel_count_probs={channels:1.0},dependency_probs={'independent':1.0},treatment_probs={'original':1.0},max_external_sources=1,allow_no_lags=True,noise=NoiseConfig(std_range=(0.02,0.05)))\ndef main():\n seed=8219; examples=8; context=32; horizon=16\n fmt=TrainingFormat(context,horizon,horizon,batch_size=examples,dtype='float32')\n plan=OnlineBatchPlan((CurriculumStage(0,make_config(6),examples),),3,seed,TrajectoryConfig(context+horizon*3,HistorySpec('uniform','uniform')),fmt)\n direct=[plan.generate(i) for i in range(3)]\n parallel=list(make_dataloader(plan,num_workers=2,prefetch_factor=1,pin_memory=False))\n assert len(parallel)==len(direct)\n for i,(a,b) in enumerate(zip(direct,parallel)):\n  for key in ('context','future','channel_mask','equation_targets','equation_mask'):\n   assert np.array_equal(getattr(a,key),b[key].numpy()),(i,key)\n  assert np.array_equal(a.noise_targets['std'],b['noise_targets']['std'].numpy())\n print('direct == two-worker batches for all 3 IDs')\nif __name__=='__main__': main()\n"
script = ROOT/'outputs'/'generator_model_evaluation'/'_pilot_07_worker_check.py'
script.parent.mkdir(parents=True, exist_ok=True)
script.write_text(worker_source,encoding='utf-8')
import subprocess,sys
result=subprocess.run([sys.executable,str(script)],capture_output=True,text=True)
print(result.stdout.strip())
if result.returncode: print(result.stderr); raise RuntimeError('worker check failed')
script.unlink()

direct == two-worker batches for all 3 IDs


## Resultado e limites

Este piloto valida a integra??o de gera??o online com DataLoader e a reprodutibilidade por ID para uma configura??o segura. Tamb?m valida a sele??o das etapas crescentes de curriculum, mas n?o testa gera??o efetiva de equa??es complexas nessa se??o.

O ambiente tem PyTorch CPU, sem CUDA. Portanto os resultados n?o estimam acelera??o, throughput de treino, stall da GPU ou compara??o justa contra dados offline. O pr?ximo experimento de performance precisa executar o mesmo modelo, batches e n?mero de updates em hardware CUDA; medir tempos de espera do loader separadamente de step time e sincronizar CUDA apenas nas regi?es de medi??o.